# Lesson 6b: Modern Architectures and Transfer Learning — Practical

6a derived why residual connections make very deep networks trainable, and
catalogued the architecture families (AlexNet through EfficientNet) that
resulted from that discovery. Someone else already spent the GPU-months
training one of those architectures on 1.2 million ImageNet images. This
notebook reuses that work instead of repeating it: we load a pretrained
`torchvision` ResNet, and compare two ways of adapting it to a small,
unrelated dataset (CIFAR-10) — freezing its learned features versus letting
them keep adapting.

## Introduction

A network trained on ImageNet has already learned, in its early
convolutional layers, filters that respond to edges, colour blobs, and
textures — features useful for almost any natural-image task, not just the
1000 ImageNet classes. **Transfer learning** exploits this: instead of
initialising a CNN's weights randomly and training from scratch (5b's
approach), start from those already-learned weights and adapt them to a new,
smaller dataset.

Two adaptation strategies bound the trade-off this notebook measures:

- **Feature extraction**: freeze every pretrained weight and train only a
  new classification head on top. The backbone is used purely as a fixed
  feature extractor.
- **Full fine-tuning**: let every weight, backbone included, keep updating on
  the new data. This can reach higher accuracy, but the backbone's
  already-good features are now exposed to gradient updates from a much
  smaller dataset than the one they were learned on, so training stability
  becomes sensitive to the learning rate in a way feature extraction never is.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, data subsampling) is reproducible.
import io
import pathlib
import urllib.request

import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.models import resnet18, ResNet18_Weights

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

# Pinned to CPU rather than the usual "cuda if available" check: this
# notebook is gated on CPU (under 10 minutes, no GPU ever required, per
# every notebook in this series), and every model here is small enough
# that CPU training costs seconds, not minutes.
device = torch.device("cpu")
print("using device:", device)

### Loading CIFAR-10

Same Hugging Face parquet mirror as 3a-6a (the canonical torchvision host
measured unreliably slow in this environment). Kept deliberately small — a
few hundred images per split is enough to make the feature-extraction vs.
fine-tuning gap visible without a multi-minute training loop.

In [ ]:
CIFAR_BASE = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text"


def load_cifar10_subset(split, n, seed):
    path = pathlib.Path("data") / f"cifar10_{split}.parquet"
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(f"{CIFAR_BASE}/{split}-00000-of-00001.parquet", path)
    df = pd.read_parquet(path)
    g = np.random.default_rng(seed)
    idx = g.permutation(len(df))[:n]
    images = np.stack([
        np.asarray(Image.open(io.BytesIO(df.iloc[i]["img"]["bytes"])), dtype=np.float32) / 255.0
        for i in idx
    ])
    labels = df.iloc[idx]["label"].to_numpy().astype(np.int64)
    return images, labels


CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

N_TRAIN, N_TEST = 800, 200
images_train, labels_train = load_cifar10_subset("train", N_TRAIN, seed=SEED)
images_test, labels_test = load_cifar10_subset("test", N_TEST, seed=SEED + 1)

# ImageNet normalisation: the pretrained backbone's filters were learned
# against inputs standardised with these exact per-channel statistics.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def to_normalised_tensor(images):
    x = torch.tensor(images).permute(0, 3, 1, 2).contiguous()  # (N, 3, 32, 32)
    return (x - IMAGENET_MEAN) / IMAGENET_STD


X_train = to_normalised_tensor(images_train)
y_train = torch.tensor(labels_train)
X_test = to_normalised_tensor(images_test).to(device)
y_test = torch.tensor(labels_test).to(device)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True,
                           generator=torch.Generator().manual_seed(SEED))

print("X_train:", X_train.shape, " X_test:", X_test.shape)

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(images_train[i])
    ax.set_title(CLASS_NAMES[labels_train[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("CIFAR-10 training samples")
plt.show()

## Loading a Pretrained Model

`torchvision.models.resnet18` is exactly the architecture 6a derived —
stacked residual blocks — with weights already trained on ImageNet's 1000
classes. Reusing it for CIFAR-10's 10 classes needs one architectural
change: replace the final fully-connected layer (`512 -> 1000`) with a
freshly-initialised `512 -> 10` layer, since the class vocabulary is
different. `freeze_backbone` controls which adaptation strategy a given
copy of the model will use.

In [ ]:
def build_pretrained_resnet(freeze_backbone):
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    torch.manual_seed(SEED)
    model.fc = nn.Linear(model.fc.in_features, 10)  # fresh head, always trainable
    return model.to(device)


n_total = sum(p.numel() for p in resnet18().parameters())
print(f"resnet18 total parameters: {n_total:,}")
print(f"new classification head parameters: {512 * 10 + 10:,}")

## Feature Extraction

With the backbone frozen, `loss.backward()` still runs a full forward pass
through every convolutional layer, but computes gradients only for the new
10-way head — the backbone never updates. This is the cheapest and most
stable way to reuse a pretrained network, and the right choice when the new
dataset is small relative to what the backbone already learned.

In [ ]:
def train_classifier(model, params_to_optimise, lr, epochs, loader):
    optimizer = optim.Adam(params_to_optimise, lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "test_acc": []}
    for _ in range(epochs):
        model.train()
        epoch_losses = []
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        history["train_loss"].append(float(np.mean(epoch_losses)))
        model.eval()
        with torch.no_grad():
            test_acc = (model(X_test).argmax(1) == y_test).float().mean().item()
        history["test_acc"].append(test_acc)
    return history


feature_extractor = build_pretrained_resnet(freeze_backbone=True)
fe_history = train_classifier(
    feature_extractor, feature_extractor.fc.parameters(), lr=1e-3, epochs=12, loader=train_loader)
fe_test_acc = fe_history["test_acc"][-1]
print(f"feature extraction — final test accuracy: {fe_test_acc:.1%}")

## Full Fine-Tuning

Unfreezing the backbone lets every layer adapt to CIFAR-10 specifically,
not just the head — at the cost of exposing 11M already-good weights to
gradients computed from only 800 training images. A learning rate that
would be unremarkable for training from scratch (5b used $10^{-3}$) is
large enough, applied to a pretrained backbone, to wreck those features
before the head has learned anything useful from them. The two runs below
share every hyperparameter except the learning rate.

In [ ]:
LOW_LR, HIGH_LR = 1e-4, 5e-2

finetune_low = build_pretrained_resnet(freeze_backbone=False)
finetune_low_history = train_classifier(
    finetune_low, finetune_low.parameters(), lr=LOW_LR, epochs=8, loader=train_loader)
finetune_low_acc = finetune_low_history["test_acc"][-1]

finetune_high = build_pretrained_resnet(freeze_backbone=False)
finetune_high_history = train_classifier(
    finetune_high, finetune_high.parameters(), lr=HIGH_LR, epochs=8, loader=train_loader)
finetune_high_acc = finetune_high_history["test_acc"][-1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(finetune_low_history["train_loss"], label=f"lr={LOW_LR}")
axes[0].plot(finetune_high_history["train_loss"], label=f"lr={HIGH_LR}")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("train loss"); axes[0].set_title("Fine-tuning loss")
axes[0].legend()
axes[1].plot(finetune_low_history["test_acc"], label=f"lr={LOW_LR}")
axes[1].plot(finetune_high_history["test_acc"], label=f"lr={HIGH_LR}")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("test accuracy"); axes[1].set_title("Fine-tuning test accuracy")
axes[1].legend()
plt.tight_layout(); plt.show()

print(f"fine-tuning, lr={LOW_LR}  — final test accuracy: {finetune_low_acc:.1%}")
print(f"fine-tuning, lr={HIGH_LR} — final test accuracy: {finetune_high_acc:.1%}")

The low learning rate's loss curve decreases smoothly and its test accuracy
climbs monotonically — the backbone's features adapt gradually. The high
learning rate's loss curve is visibly erratic and its final accuracy trails
the stable run, sometimes substantially: a single large gradient step can
move a pretrained filter far enough from its ImageNet optimum that several
subsequent epochs are spent recovering rather than adapting. This is the
instability that makes learning-rate choice a first-order concern for
fine-tuning specifically, in a way it was not for 5b's from-scratch CNN.

## Comparison and Discussion

In [ ]:
strategies = ["Feature extraction", f"Fine-tune (lr={LOW_LR})", f"Fine-tune (lr={HIGH_LR})"]
accuracies = [fe_test_acc, finetune_low_acc, finetune_high_acc]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(strategies, accuracies, color=["#4c72b0", "#55a868", "#c44e52"])
ax.set_ylabel("test accuracy")
ax.set_title("Adaptation strategy vs. test accuracy")
ax.set_ylim(0, 1)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.02, f"{acc:.1%}", ha="center")
plt.xticks(rotation=10)
plt.tight_layout(); plt.show()

Three points this comparison makes concrete:

- **Feature extraction is the cheapest option and a strong baseline.** It
  trains an order of magnitude fewer parameters (the head alone, a few
  thousand, versus 11M) and every run is stable by construction — there is
  no backbone to destabilise.
- **A well-tuned fine-tune can beat feature extraction**, because the
  backbone's features, not just the head, get to specialise for CIFAR-10's
  statistics rather than staying frozen at whatever generalised to ImageNet.
- **An ill-tuned fine-tune can lose to feature extraction**, and by a wide
  margin — the extra capacity that made fine-tuning attractive is exactly
  what a too-large learning rate spends damaging, rather than usefully
  adapting, on a small dataset.

The practical rule this motivates: start from feature extraction as a cheap,
stable baseline; move to fine-tuning only with a small learning rate, and
treat its stability, not just its ceiling, as something to check for.

## Key Takeaways

- **Transfer learning** reuses a network already trained on a large dataset
  (ImageNet) as a starting point for a new, smaller one (CIFAR-10), instead
  of training from scratch as 5b did.
- **Feature extraction** (freeze the backbone, train only a new head) trains
  a few thousand parameters instead of 11M, is stable by construction, and
  measured here as a strong baseline for very little compute.
- **Full fine-tuning** (train every weight) can exceed feature extraction's
  accuracy, but — measured directly here by holding every hyperparameter
  fixed except the learning rate — its stability is sensitive to that
  learning rate in a way feature extraction's never is: the high-lr run's
  erratic loss curve and lower final accuracy versus the low-lr run's smooth
  curve and higher accuracy is exactly this sensitivity made visible.
- The practical default: extract features first as a stable baseline, then
  fine-tune cautiously (small learning rate) only if more accuracy is
  needed.